**Import Datasets and Packages**

In [1]:
import pandas as pd
from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


In [2]:
userdata.get("HF_TOKEN")

#ds  = load_dataset("mpingale/mental-health-chat-dataset", token="HF_TOKEN")

#For this week, I will be using only the model below
ds2 = load_dataset("daniel2588/sarcasm", token="HF_TOKEN")

**Exploring Data**

In [3]:
df_sarcasm = ds2["train"].to_pandas()

print(f"Total rows: {len(df_sarcasm)}")
print(f"\nLabel distribution:")
print(df_sarcasm["label"].value_counts())
print(f"\nSample sarcastic comment:")
print(df_sarcasm[df_sarcasm["label"]==1]["comment"].iloc[0])
print(f"\nSample non-sarcastic comment:")
print(df_sarcasm[df_sarcasm["label"]==0]["comment"].iloc[0])

Total rows: 1010826

Label distribution:
label
0    505413
1    505413
Name: count, dtype: int64

Sample sarcastic comment:
But they'll have all those reviews!

Sample non-sarcastic comment:
NC and NH.


**Shorten the Dataset**

- It would take a really long time to run the transformer on all 1M+ rows for sarcasm, instead, I will focus on 50K samples. So I am spitting the data further to have 25K for Sarcasm POS and 25K for Sarcasm NEG

In [4]:
df_sarcasm = ds2["train"].to_pandas()

# Take 25k from each label = 50k balanced
df_zeros = df_sarcasm[df_sarcasm["label"] == 0].sample(25000, random_state=42)
df_ones  = df_sarcasm[df_sarcasm["label"] == 1].sample(25000, random_state=42)

# Combine and shuffle
df_sample = pd.concat([df_zeros, df_ones]).sample(frac=1, random_state=42).reset_index(drop=True)

# Verify balance
print(f"Label distribution:\n{df_sample['label'].value_counts()}")


Label distribution:
label
1    25000
0    25000
Name: count, dtype: int64


In [5]:
train_df, test_df = train_test_split(df_sample, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df[["comment", "label"]].reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df[["comment", "label"]].reset_index(drop=True))

print(f"\nTrain size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")


Train size: 40000
Test size:  10000


In [6]:
# Clean before tokenizing
train_dataset = train_dataset.filter(lambda x: x["comment"] is not None and len(str(x["comment"])) > 0)
test_dataset  = test_dataset.filter(lambda x: x["comment"] is not None and len(str(x["comment"])) > 0)

print(f"Train size after cleaning: {len(train_dataset)}")
print(f"Test size after cleaning:  {len(test_dataset)}")

Filter:   0%|          | 0/40000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Train size after cleaning: 39998
Test size after cleaning:  10000


**Tokenize**

In [12]:
# DistilBERT — small, fast, perfect for classification (see justifications.md)
MODEL_NAME = "distilbert-base-uncased"
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    comments = [str(c) if c is not None else "" for c in batch["comment"]]
    return tokenizer_bert(
        batch["comment"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_tokenized = train_dataset.map(tokenize, batched=True)
test_tokenized  = test_dataset.map(tokenize, batched=True)

print("\n✅ All done — ready for training")

Map:   0%|          | 0/39998 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]


✅ All done — ready for training


In [15]:
print(df_sample[["comment", "label", "score"]].head(10))
print(f"\nScore range: {df_sample['score'].min()} to {df_sample['score'].max()}")

                                             comment  label  score
0               Not even a terabyte worth of storage      1      1
1                                           pervert.      0      3
2  You figure there would be a 5th jet for symmet...      0      3
3                                                 rt      0      2
4                              You had me at DT drop      1      1
5         Thanks Samsung for you wonderful batteries      1      3
6                           What's funny about that?      0      6
7  Oh you mean the son of Lee Kwan Yiew is paid a...      1      1
8    Throw in some tendies and we'll call it a deal.      0      1
9  yeah i'm soo jealous of your tiny girl like th...      1      1

Score range: -298 to 3721


**Training the Classifier**

In [9]:
# Load DistilBERT for binary classification (sarcastic or not)
classifier_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# How we measure performance
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/sarcasm-classifier",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=classifier_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

trainer.train()
print("✅ Sarcasm classifier trained")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.541646,0.540483,0.730800,0.730111
2,0.431339,0.573141,0.726800,0.724630
3,0.244148,0.742821,0.722300,0.722178


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


✅ Sarcasm classifier trained


**Evaluate and Save to Drive**

In [10]:
# Final evaluation
results = trainer.evaluate()
print(f"\n=== Results ===")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Score: {results['eval_f1']:.4f}")

# Save to Drive
trainer.save_model("/content/drive/MyDrive/sarcasm-classifier/final")
tokenizer_bert.save_pretrained("/content/drive/MyDrive/sarcasm-classifier/final")
print("✅ Classifier saved to Drive")


=== Results ===
Accuracy: 0.7305
F1 Score: 0.7298


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Classifier saved to Drive


**Mini Test**

In [16]:
sarcasm_detector = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/sarcasm-classifier/final"
)

test_inputs = [
    "I'm really struggling today and feel so alone",
    "Oh great, another wonderful day of everything going wrong",
    "I feel anxious all the time and don't know why",
    "Yeah sure, therapy is definitely going to fix all my problems"
]

print("=== Sarcasm Detection Test ===")
for text in test_inputs:
    result = sarcasm_detector(text)[0]
    label  = "🙄 Sarcastic" if result["label"] == "LABEL_1" else "💬 Genuine"
    print(f"{label} ({result['score']:.2f}): {text}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== Sarcasm Detection Test ===
💬 Genuine (0.67): I'm really struggling today and feel so alone
🙄 Sarcastic (0.91): Oh great, another wonderful day of everything going wrong
💬 Genuine (0.79): I feel anxious all the time and don't know why
🙄 Sarcastic (0.97): Yeah sure, therapy is definitely going to fix all my problems
